Scenario 1: A single data scientist participating in an ML competition
MLflow setup:

- Tracking server: no
- Backend store: local filesystem
- Artifacts store: local filesystem
The experiments can be explored locally by launching the MLflow UI.

In [10]:
import mlflow

In [15]:
MLFLOW_TRACKING_URI="file:/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"tracking URI: {mlflow.get_tracking_uri()}")

tracking URI: file:/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns


### Up server UI with the comand:
 mlflow ui \
    --backend-store-uri file:/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns \
    --host 127.0.0.1 \
    --port 5000


In [31]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns/667205099806723233', creation_time=1778189801162, experiment_id='667205099806723233', last_update_time=1778189801162, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='file:///home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns/0', creation_time=1778189493771, experiment_id='0', last_update_time=1778189493771, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

#### Creating an experiment and logging a new run

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.2, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/05/07 16:50:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 16:50:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


default artifacts URI: 'file:///home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns/667205099806723233/cbdbd1360bf942eab19d3ad786526df6/artifacts'


In [33]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns/667205099806723233', creation_time=1778189801162, experiment_id='667205099806723233', last_update_time=1778189801162, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='file:///home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/mlruns/0', creation_time=1778189493771, experiment_id='0', last_update_time=1778189493771, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Interacting with the model registry

In [20]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [30]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
    client.search_experiments()
    experiment = client.get_experiment_by_name("my-experiment-1")
    runs = client.search_runs([experiment.experiment_id])

    for run in runs:
        print(f"Run ID: {run.info.run_id}, Accuracy: {run.data.metrics.get('accuracy')}, Params: {run.data.params}")
except MlflowException:
    print("It's not possible to access the model registry :(")

Run ID: 92d50193b35b40aeab68a6e8b2465ff4, Accuracy: 0.9666666666666667, Params: {'random_state': '42', 'C': '0.2'}
Run ID: 097cfd17e7244249873f972a47bf74bb, Accuracy: 0.96, Params: {'random_state': '42', 'C': '0.1'}
Run ID: 7b37e90cab6d402882057ec318e91c5d, Accuracy: 0.96, Params: {'C': '0.1', 'random_state': '42'}
